# Spline-factorization correction of the MCMC chains: before/after comparison

PROfit builds the predicted spectrum at $\eta=(\eta_1,\eta_2,\dots)$ as a *product* of independent one-dimensional splines, one per z-expansion PCA parameter. The true per-bin response is an exact multivariate quadratic in $\eta$ with cross terms $\eta_i\eta_j$; the product replaces the true cross term by a spurious $b_1b_2$ term (about four times larger, with the wrong sign structure). `12_spline_factorization_validation.ipynb` tabulated the resulting error in the data term of the $\chi^2$,

$$\Delta\chi^2_{\rm data}(\eta)=\chi^2_{\rm data}\big(\mu^{\rm spline}(\eta)\big)-\chi^2_{\rm data}\big(\mu^{\rm exact}(\eta)\big),$$

on grids in $\eta$ space for every fit and suite and saved them to `tables/spline_factorization/grids/`. Because the Gaussian pull term $\sum_i\eta_i^2$ is identical in both models, the ratio of the exact to the sampled posterior density at a chain sample is

$$w(\eta)=\exp\!\big(+\Delta\chi^2_{\rm data}(\eta)/2\big),$$

so every stored chain is corrected by importance reweighting, with no fit re-run. `python/scripts/spline_reweighting.py` implements the grid interpolation and the weights; `postfit_physical_parameters.load_fit` applies them to every z-expansion fit, so all analysis notebooks quote the corrected posteriors. This notebook demonstrates the correction for two illustrative cases:

- **Case 1, Gaussian prior:** `minerva_k6` on NuWro fake data. The chain is confined to the knot range $|\eta_i|\le3$, where $|\Delta\chi^2_{\rm data}|\lesssim0.1$ inside the 95% region; the correction should be negligible (posterior-mean shift $<0.01\sigma$, width change $\lesssim1\%$) and the before/after contours should overlap.
- **Case 2, uniform prior:** `minerva_k6_uniform` on open data. The posterior sits at $\eta_2\approx-7.5$, deep in PROfit's extrapolation region, where the factorized error grows quartically; the corrected contours should be visibly shifted and narrowed (mean shift $\approx0.05\sigma$, width change $\approx9\%$).

Blue is always the stored chain (factorized splines), orange the chain reweighted to the exact response. Figures are written below `figs/spline_reweighting/`, tables below `tables/spline_reweighting/`.

In [ ]:
from pathlib import Path
import sys
import warnings

import corner
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import numpy as np
import pandas as pd
from IPython.display import display

# Locate ma_zexp/python/scripts whether the notebook starts here or from axial_mass.
start = Path.cwd().resolve()
helper_dir = next(
    (parent / 'ma_zexp' / 'python' / 'scripts' for parent in (start, *start.parents)
     if (parent / 'ma_zexp' / 'python' / 'scripts' / 'postfit_physical_parameters.py').is_file()),
    None,
)
if helper_dir is None:
    raise FileNotFoundError('Could not locate ma_zexp/python/scripts/postfit_physical_parameters.py')
if str(helper_dir) not in sys.path:
    sys.path.insert(0, str(helper_dir))

from postfit_physical_parameters import FIGURE_ROOT, PUBLICATION_RC, SPECS, load_fit, prior_label
from spline_reweighting import (
    SUITE_FIT_RESULTS, compute_importance_weights, describe_ess, eta_to_a_coefficients, load_chain,
    load_dchi2_grid, pareto_khat, weighted_quantile, weighted_summary, zexp_affine_map,
)
from zexp_reweighting import axial_form_factor_zexp   # uboone_ngem/src is put on sys.path by postfit_physical_parameters

mpl.rcParams.update(PUBLICATION_RC)
np.set_printoptions(linewidth=160, precision=5, suppress=True)
pd.set_option('display.width', 250)
pd.set_option('display.max_columns', 40)

REPO_MA_ZEXP = helper_dir.parents[1]
FIG_DIR = FIGURE_ROOT / 'spline_reweighting'
TABLE_DIR = REPO_MA_ZEXP / 'tables' / 'spline_reweighting'
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)
SAVE_OUTPUTS = True
SAVE_DPI = 600
CREDIBLE_LEVELS = (0.68, 0.95)

# Okabe-Ito, as in the other publication figures: blue = stored chain, vermillion = reweighted.
COLOR_BEFORE, COLOR_AFTER, COLOR_NOTE = '#0072B2', '#D55E00', '#596273'
LABEL_BEFORE = 'PROfit chain (factorized splines)'
LABEL_AFTER = 'Reweighted to the exact response'
SUITE_LABEL = {'nuwro': 'NuWro fake data', 'asimov': 'Asimov', 'opendata': 'Open data'}
SPEC_BY_KEY = {spec.key: spec for spec in SPECS}

CASES = {
    'gaussian': dict(fit='minerva_k6', suite='nuwro', title='Case 1: Gaussian prior', stem='gaussian_minerva_k6_nuwro'),
    'uniform': dict(fit='minerva_k6_uniform', suite='opendata', title='Case 2: uniform prior', stem='uniform_minerva_k6_opendata'),
}
for case in CASES.values():
    case['spec'] = SPEC_BY_KEY[case['fit']]
    case['label'] = f"{prior_label(case['spec'])} | {SUITE_LABEL[case['suite']]}"
print('ma_zexp repo:', REPO_MA_ZEXP)
print('figures     :', FIG_DIR)
print('tables      :', TABLE_DIR)

## Section 1: setup and weight diagnostics

For each case the chain (`*_mcmc_chain` tree of the PROfile file, columns `weight_spline_FAzexpMinervaK6PCA1/2`) and the saved $\Delta\chi^2_{\rm data}$ grid are loaded; `load_dchi2_grid` picks the extended $[-10,10]^2$ grid for the uniform-prior fit (its chain roams outside the knot range) and the full $[-3,3]^2$ grid otherwise. $\Delta\chi^2_{\rm data}$ is interpolated linearly onto every sample and $w_k=e^{+\Delta\chi^2_{\rm data}/2}$, normalized to unit sum. The effective sample size ${\rm ESS}=(\sum_k w_k)^2/\sum_k w_k^2$ measures how much sample quality the reweighting costs.

In [ ]:
CASE_DATA = {}
diagnostic_rows = []
for name, case in CASES.items():
    chain_df, eta = load_chain(case['fit'], case['suite'])
    if chain_df is None:
        raise FileNotFoundError(f"no chain for {case['fit']} / {case['suite']}")
    grid = load_dchi2_grid(case['fit'], case['suite'])      # 'auto': extended for uniform-prior fits, full otherwise
    assert tuple(grid['eta_names']) == tuple(case['spec'].prior.variation_branches)
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always')
        weights_raw, weights_norm, ess, dchi2 = compute_importance_weights(eta, grid)
    n = len(eta)
    relative = weights_raw / weights_raw.mean()
    print(f"{case['title']}: {case['label']}")
    print(f"    chain: {chain_df.attrs['path']}  ({n:,} samples, columns {list(chain_df.columns)})")
    print(f"    grid : {grid['path'].name}  ({grid['grid_kind']}, half-range {grid['half_range']:.0f}, step {grid['step']}, "
          f"max |dchi2| on the grid {np.abs(grid['dchi2']).max():.3g})")
    print('    ' + describe_ess(ess, n))
    print(f"    dchi2_data at the samples: min {dchi2.min():+.4f}, max {dchi2.max():+.4f}, mean {dchi2.mean():+.5f}; "
          f"relative weights w/<w>: min {relative.min():.4f}, max {relative.max():.4f}, std {relative.std():.4f}")
    print(f"    samples with some |eta_i| > 3: {100 * np.mean(np.any(np.abs(eta) > 3, axis=1)):.1f}%; "
          f"module warnings: {[str(c.message) for c in caught] or 'none'}")
    CASE_DATA[name] = dict(case, chain_df=chain_df, eta=eta, grid=grid, weights_raw=weights_raw, weights_norm=weights_norm,
                           ess=ess, dchi2=dchi2, n=n, relative=relative,
                           eta_labels=[rf'$\eta_{i + 1}$' for i in range(eta.shape[1])],
                           eta_names=[f'eta{i + 1}' for i in range(eta.shape[1])])
    diagnostic_rows.append({'case': case['title'], 'fit': case['fit'], 'suite': case['suite'], 'grid': grid['grid_type'],
                            'N': n, 'ESS': ess, 'ESS / N': ess / n, 'min dchi2': dchi2.min(), 'max dchi2': dchi2.max(),
                            'max |dchi2|': np.abs(dchi2).max(), 'min w/<w>': relative.min(), 'max w/<w>': relative.max(),
                            'fraction |eta|>3': np.mean(np.any(np.abs(eta) > 3, axis=1))})
diagnostics = pd.DataFrame(diagnostic_rows).set_index('case')
display(diagnostics.style.format({'ESS': '{:.0f}', 'ESS / N': '{:.4f}', 'min dchi2': '{:+.4f}', 'max dchi2': '{:+.4f}',
                                  'max |dchi2|': '{:.4f}', 'min w/<w>': '{:.4f}', 'max w/<w>': '{:.4f}', 'fraction |eta|>3': '{:.3f}'}))
if SAVE_OUTPUTS:
    diagnostics.to_csv(TABLE_DIR / 'weight_diagnostics.csv')

### Implementation check

For the Gaussian/NuWro case the factorization error inside the sampled region is small by construction ($|\Delta\chi^2_{\rm data}|<1$ everywhere the chain went) and the weights are nearly uniform (${\rm ESS}>0.95\,N$). If either assertion fails, the grid, the chain columns or the interpolation are wrong.

In [ ]:
gaussian = CASE_DATA['gaussian']
max_abs_dchi2 = float(np.abs(gaussian['dchi2']).max())
assert max_abs_dchi2 < 1.0, f"Gaussian/NuWro: max |dchi2_data| at the samples is {max_abs_dchi2:.3f} >= 1"
assert gaussian['ess'] > 0.95 * gaussian['n'], f"Gaussian/NuWro: ESS = {gaussian['ess']:.0f} <= 0.95 N = {0.95 * gaussian['n']:.0f}"
print(f"Gaussian/NuWro case: max |dchi2_data| at the samples = {max_abs_dchi2:.3f} (< 1), "
      f"ESS / N = {gaussian['ess'] / gaussian['n']:.4f} (> 0.95). Implementation checks pass.")

### Distribution of $\Delta\chi^2_{\rm data}$ and of the relative weights across the chains

Left: $\Delta\chi^2_{\rm data}$ interpolated onto every chain sample. Right: the relative weight $w_k/\langle w\rangle$. For the Gaussian case both are tightly concentrated ($\Delta\chi^2_{\rm data}$ near zero, weights near one); for the uniform-prior case they spread out.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11.5, 7.6), constrained_layout=True)
for row, (name, data) in enumerate(CASE_DATA.items()):
    ax_d, ax_w = axes[row]
    ax_d.hist(data['dchi2'], bins=120, color=COLOR_BEFORE, alpha=0.85, lw=0)
    ax_d.axvline(0, color='0.4', lw=0.8)
    ax_d.set_xlabel(r'$\Delta\chi^2_{\rm data}(\eta^{(k)})$')
    ax_d.set_ylabel('Chain samples')
    ax_d.set_yscale('log')
    ax_d.text(0.03, 0.96, f"{data['title']}\n{data['label']}\n"
              rf"min {data['dchi2'].min():+.3f}, max {data['dchi2'].max():+.3f}, mean {data['dchi2'].mean():+.4f}",
              transform=ax_d.transAxes, ha='left', va='top', fontsize=10, color=COLOR_NOTE)
    ax_w.hist(data['relative'], bins=120, color=COLOR_AFTER, alpha=0.85, lw=0)
    ax_w.axvline(1, color='0.4', lw=0.8)
    ax_w.set_xlabel(r'$w_k / \langle w \rangle$')
    ax_w.set_ylabel('Chain samples')
    ax_w.set_yscale('log')
    ax_w.text(0.97, 0.96, f"ESS / N = {data['ess'] / data['n']:.4f}\n"
              rf"min {data['relative'].min():.3f}, max {data['relative'].max():.3f}, std {data['relative'].std():.4f}",
              transform=ax_w.transAxes, ha='right', va='top', fontsize=10, color=COLOR_NOTE)
    for ax in (ax_d, ax_w):
        ax.tick_params(which='both', direction='in', top=True, right=True)
if SAVE_OUTPUTS:
    fig.savefig(FIG_DIR / 'weight_diagnostics.pdf', dpi=SAVE_DPI, bbox_inches='tight', pad_inches=0.03, facecolor='white')
    print('Saved:', FIG_DIR / 'weight_diagnostics.pdf')
display(fig)
plt.close(fig)

## Section 2: corner plots in $\eta$ space

Each panel overlays the stored chain (blue, solid) and the reweighted chain (orange, dashed): contours enclosing 68% and 95% of the (weighted) samples off the diagonal, normalized marginal histograms on the diagonal, and the posterior means as markers. The `corner` package is used with its `weights=` argument; the weights are scaled by $N$ so that the weighted and unweighted histograms share the same normalization.

In [ ]:
def overlay_corner(samples, weights_norm, labels, stem, title, subtitle, bins=60, smooth=1.0):
    n_par = samples.shape[1]
    low, high = samples.min(axis=0), samples.max(axis=0)
    pad = 0.04 * (high - low)
    ranges = [(l - p, h + p) for l, h, p in zip(low, high, pad)]
    common = dict(bins=bins, range=ranges, levels=CREDIBLE_LEVELS, smooth=smooth, labels=labels,
                  plot_datapoints=False, plot_density=False, fill_contours=False, no_fill_contours=True,
                  label_kwargs=dict(fontsize=15), max_n_ticks=5)
    # A larger canvas than corner's default leaves the empty upper triangle free for the title and legend.
    fig = plt.figure(figsize=(4.0 * n_par, 4.0 * n_par))
    corner.corner(samples, color=COLOR_BEFORE, fig=fig, contour_kwargs=dict(linewidths=1.6, linestyles='-'),
                  hist_kwargs=dict(density=True, color=COLOR_BEFORE, lw=1.6), **common)
    corner.corner(samples, weights=weights_norm * len(weights_norm), color=COLOR_AFTER, fig=fig,
                  contour_kwargs=dict(linewidths=1.6, linestyles='--'),
                  hist_kwargs=dict(density=True, color=COLOR_AFTER, lw=1.6, ls='--'), **common)
    axes = np.array(fig.axes).reshape(n_par, n_par)
    mean_before = samples.mean(axis=0)
    mean_after = np.average(samples, weights=weights_norm, axis=0)
    for i in range(n_par):
        for j in range(i):
            axes[i, j].plot(mean_before[j], mean_before[i], marker='+', ls='none', color=COLOR_BEFORE, ms=11, mew=1.8, zorder=6)
            axes[i, j].plot(mean_after[j], mean_after[i], marker='x', ls='none', color=COLOR_AFTER, ms=8, mew=1.8, zorder=6)
        axes[i, i].axvline(mean_before[i], color=COLOR_BEFORE, lw=0.9, alpha=0.7)
        axes[i, i].axvline(mean_after[i], color=COLOR_AFTER, lw=0.9, ls='--', alpha=0.9)
        for ax in axes[i]:
            ax.tick_params(which='both', direction='in', top=True, right=True, labelsize=10)
    handles = [Line2D([], [], color=COLOR_BEFORE, lw=1.6, label=LABEL_BEFORE),
               Line2D([], [], color=COLOR_AFTER, lw=1.6, ls='--', label=LABEL_AFTER),
               Line2D([], [], color=COLOR_BEFORE, marker='+', ls='none', ms=10, mew=1.8, label='posterior mean, before'),
               Line2D([], [], color=COLOR_AFTER, marker='x', ls='none', ms=8, mew=1.8, label='posterior mean, after')]
    # Upper-right (empty) part of the corner grid: title and legend.
    top_right = axes[0, -1].get_position()
    heading = '\n'.join([title, *subtitle.split(' | '), '68% and 95% credible contours'])
    fig.text(top_right.x1, top_right.y1, heading, ha='right', va='top', fontsize=11, color='black',
             linespacing=1.35, transform=fig.transFigure)
    fig.legend(handles=handles, loc='upper right', bbox_to_anchor=(top_right.x1, top_right.y1 - 0.135), fontsize=10,
               frameon=False, handlelength=2.2)
    if SAVE_OUTPUTS:
        fig.savefig(FIG_DIR / f'{stem}.pdf', dpi=SAVE_DPI, bbox_inches='tight', pad_inches=0.03, facecolor='white')
        print('Saved:', FIG_DIR / f'{stem}.pdf')
    display(fig)
    plt.close(fig)


for name, data in CASE_DATA.items():
    overlay_corner(data['eta'], data['weights_norm'], data['eta_labels'], f"corner_eta_{data['stem']}",
                   data['title'], data['label'])

**Expected result.** Case 1: the two sets of contours coincide (the mean shift is below $0.01\sigma$). Case 2: the reweighted contours are shifted and narrowed relative to the stored chain, most visibly along $\eta_2$, where the posterior sits around $-7.5$ and PROfit extrapolates its outermost spline segments.

## Section 3: corner plots in $z$-expansion coefficient space

The chain is mapped to the complete coefficient vector with the exact affine map $a(\eta)=a^{\rm CV}+\sum_i\eta_i\,\Delta a_i$, where $a^{\rm CV}$ is the prior's central vector and $\Delta a_i$ the completed shift (dependent coefficients re-solved from the four sum rules and $F_A(0)$) for one standard deviation along PCA direction $i$; `zexp_affine_map` builds it with the same PCA convention as the stored PROfit branches. The result is checked against `postfit_physical_parameters.load_fit`, which performs the same transformation and applies the same weights to every fit in the analysis. The corner plots show the free coefficients $a_1,a_2$ (the physically meaningful representation; $a_0$ and $a_3,\dots,a_6$ follow from the constraints).

In [ ]:
for name, data in CASE_DATA.items():
    prior = data['spec'].prior
    a_cv, delta_a = zexp_affine_map(prior)                      # a_cv: (N_coeffs,), delta_a: (N_pca, N_coeffs)
    a_samples = eta_to_a_coefficients(data['eta'], a_cv, delta_a)
    # The library must produce the same coefficient samples and the same weights.
    result = load_fit(data['spec'], SUITE_FIT_RESULTS[data['suite']])
    assert result is not None and np.allclose(result['samples'], a_samples, rtol=0, atol=1e-9)
    assert np.allclose(result['weights'], data['weights_norm'], rtol=1e-9, atol=0)
    free = np.arange(1, len(prior.free_a_values) + 1)
    data.update(a_cv=a_cv, delta_a=delta_a, a_samples=a_samples, free_indices=free,
                a_labels=[rf'$a_{k}$' for k in free], a_names=[f'a{k}' for k in free], result=result)
    print(f"{data['title']}: {prior.name} (kmax={prior.kmax}, t0={prior.t0_gev2} GeV^2, t_cut={prior.t_cut_gev2:.6f} GeV^2, "
          f"F_A(0)={prior.fa_q2_zero}); load_fit reproduces a(eta) and the weights.")
    affine = pd.DataFrame(np.vstack([a_cv, delta_a]), index=['a_CV'] + [f'delta a for {e}' for e in data['eta_names']],
                          columns=[f'a{k}' for k in range(len(a_cv))])
    display(affine.style.format('{:+.5f}').set_caption(f"{data['title']}: affine map from eta to the coefficients"))
    overlay_corner(a_samples[:, free], data['weights_norm'], data['a_labels'], f"corner_a_{data['stem']}",
                   data['title'], data['label'])

## Section 4: numerical summary

Per case and parameter: the posterior mean before (unweighted chain) and after (importance-weighted), the 68% half-width before and after (half the length of the equal-tailed 16-84% interval), the shift of the mean in units of the before-$\sigma$ (standard deviation of the unweighted chain), and the relative change of the half-width and of the standard deviation. Both the fit parameters $\eta_i$ and the free coefficients $a_j$ are listed.

In [ ]:
def before_after_table(samples, weights_norm, names):
    before = weighted_summary(samples, None, CREDIBLE_LEVELS, names)
    after = weighted_summary(samples, weights_norm, CREDIBLE_LEVELS, names)
    return pd.DataFrame({
        'mean (before)': before['mean'], 'mean (after)': after['mean'],
        '68% half-width (before)': before['half_width_68'], '68% half-width (after)': after['half_width_68'],
        'shift / sigma(before)': (after['mean'] - before['mean']) / before['std'],
        'half-width change [%]': 100 * (after['half_width_68'] / before['half_width_68'] - 1),
        'std (before)': before['std'], 'std (after)': after['std'],
        'std change [%]': 100 * (after['std'] / before['std'] - 1),
        '95% half-width (before)': before['half_width_95'], '95% half-width (after)': after['half_width_95'],
    }, index=names)


FORMAT = {c: '{:+.4f}' for c in ('mean (before)', 'mean (after)', 'shift / sigma(before)')} | \
         {c: '{:.4f}' for c in ('68% half-width (before)', '68% half-width (after)', 'std (before)', 'std (after)',
                                '95% half-width (before)', '95% half-width (after)')} | \
         {'half-width change [%]': '{:+.2f}', 'std change [%]': '{:+.2f}'}
for name, data in CASE_DATA.items():
    samples = np.column_stack([data['eta'], data['a_samples'][:, data['free_indices']]])
    names = data['eta_names'] + data['a_names']
    table = before_after_table(samples, data['weights_norm'], names)
    table.index.name = 'parameter'
    data['table'] = table
    display(table.style.format(FORMAT).set_caption(
        f"{data['title']}: {data['label']} | ESS/N = {data['ess'] / data['n']:.4f}"))
    if SAVE_OUTPUTS:
        table.to_csv(TABLE_DIR / f"before_after_{data['stem']}.csv")
        print('Saved:', TABLE_DIR / f"before_after_{data['stem']}.csv")

g, u = CASE_DATA['gaussian']['table'], CASE_DATA['uniform']['table']
g_shift, g_width = g['shift / sigma(before)'].abs().max(), g['std change [%]'].abs().max()
u_shift, u_width = u['shift / sigma(before)'].abs().max(), u['std change [%]'].abs().max()
print(f"Case 1 (Gaussian, NuWro):   largest |mean shift| = {g_shift:.4f} sigma, largest |width change| = {g_width:.2f}%")
print(f"Case 2 (uniform, open data): largest |mean shift| = {u_shift:.4f} sigma, largest |width change| = {u_width:.2f}%")
assert g_shift < 0.01, f'Gaussian/NuWro case shifts by {g_shift:.4f} sigma >= 0.01 sigma'
assert u_shift > g_shift and u_width > g_width, 'the uniform-prior case should be affected more than the Gaussian one'

## Section 5: $F_A(Q^2)$ before and after

The coefficient samples are propagated to $F_A(Q^2)=\sum_k a_k z^k$ on a $Q^2$ grid, and the pointwise median and central 68% band are computed without weights (stored chain) and with the importance weights (exact response). The lower panels show the reweighted median and band edges divided by the median of the stored chain. For the Gaussian case the two bands are indistinguishable; for the uniform-prior case the difference is visible.

In [ ]:
Q2 = np.geomspace(0.01, 2.0, 300)
Q2_TICKS = ([0.01, 0.05, 0.1, 0.5, 1.0, 2.0], ['0.01', '0.05', '0.1', '0.5', '1', '2'])
MAX_CURVES = 100_000


def fa_curves(a_samples, q2, prior):
    z = ((np.sqrt(prior.t_cut_gev2 + q2) - np.sqrt(prior.t_cut_gev2 - prior.t0_gev2))
         / (np.sqrt(prior.t_cut_gev2 + q2) + np.sqrt(prior.t_cut_gev2 - prior.t0_gev2)))
    return a_samples @ np.vander(z, N=a_samples.shape[1], increasing=True).T


fig, axes = plt.subplots(2, 2, figsize=(12.5, 7.6), sharex='col',
                         gridspec_kw=dict(height_ratios=(3, 1.25), hspace=0.06, wspace=0.2))
for col, (name, data) in enumerate(CASE_DATA.items()):
    prior = data['spec'].prior
    # Vectorized evaluation must agree with the shared evaluator on the central value.
    assert np.allclose(fa_curves(data['a_cv'][None, :], Q2, prior)[0],
                       axial_form_factor_zexp(Q2, data['a_cv'], prior.t0_gev2, prior.t_cut_gev2), rtol=0, atol=1e-12)
    rng = np.random.default_rng(2026)
    index = rng.choice(data['n'], size=min(data['n'], MAX_CURVES), replace=False)
    curves = fa_curves(data['a_samples'][index], Q2, prior)
    w = data['weights_norm'][index]
    w = w / w.sum()
    q_before = weighted_quantile(curves, None, [0.16, 0.50, 0.84])
    q_after = weighted_quantile(curves, w, [0.16, 0.50, 0.84])
    data['fa_before'], data['fa_after'] = q_before, q_after
    ax, ratio = axes[0, col], axes[1, col]
    # Conventional negative F_A drawn as the positive quantity -F_A: reverse the bounds when negating.
    ax.fill_between(Q2, -q_before[2], -q_before[0], color=COLOR_BEFORE, alpha=0.25, lw=0)
    ax.plot(Q2, -q_before[1], color=COLOR_BEFORE, lw=2)
    ax.fill_between(Q2, -q_after[2], -q_after[0], facecolor='none', edgecolor=COLOR_AFTER, hatch='///', lw=0)
    ax.plot(Q2, -q_after[2], color=COLOR_AFTER, lw=1.0, ls='--')
    ax.plot(Q2, -q_after[0], color=COLOR_AFTER, lw=1.0, ls='--')
    ax.plot(Q2, -q_after[1], color=COLOR_AFTER, lw=2, ls='--')
    ax.set_ylabel(r'$-F_A(Q^2)$')
    ax.set_title(f"{data['title']}: {data['label']}", fontsize=11.5)
    denominator = q_before[1]
    ratio.fill_between(Q2, q_after[0] / denominator, q_after[2] / denominator, facecolor='none', edgecolor=COLOR_AFTER, hatch='///', lw=0)
    ratio.fill_between(Q2, q_before[0] / denominator, q_before[2] / denominator, color=COLOR_BEFORE, alpha=0.25, lw=0)
    ratio.plot(Q2, q_after[1] / denominator, color=COLOR_AFTER, lw=2, ls='--')
    ratio.axhline(1, color=COLOR_BEFORE, lw=1.2)
    ratio.set_ylabel('Ratio to median\n(stored chain)', fontsize=11)
    ratio.set_xlabel(r'$Q^2$ [GeV$^2$]')
    largest = np.max(np.abs(q_after[1] / denominator - 1))
    ratio.text(0.02, 0.06, f'largest median shift: {100 * largest:.2f}%', transform=ratio.transAxes, ha='left', va='bottom',
               fontsize=10, color=COLOR_NOTE)
    for axis in (ax, ratio):
        axis.set_xscale('log')
        axis.set_xlim(Q2[0], Q2[-1])
        axis.set_xticks(Q2_TICKS[0], labels=Q2_TICKS[1])
        axis.grid(which='major', color='#9AA4B2', alpha=0.22, linewidth=0.7)
        axis.tick_params(which='both', direction='in', top=True, right=True)
handles = [(Patch(facecolor=mpl.colors.to_rgba(COLOR_BEFORE, 0.25), edgecolor='none'), Line2D([], [], color=COLOR_BEFORE, lw=2)),
           (Patch(facecolor='none', edgecolor=COLOR_AFTER, hatch='///'), Line2D([], [], color=COLOR_AFTER, lw=2, ls='--'))]
from matplotlib.legend_handler import HandlerTuple
axes[0, 0].legend(handles, [LABEL_BEFORE + ', median and 68% band', LABEL_AFTER + ', median and 68% band'],
                  handler_map={tuple: HandlerTuple(ndivide=1)}, loc='lower left', fontsize=9.5, frameon=False, handlelength=2.8)
if SAVE_OUTPUTS:
    fig.savefig(FIG_DIR / 'fa_band_comparison.pdf', dpi=SAVE_DPI, bbox_inches='tight', pad_inches=0.03, facecolor='white')
    print('Saved:', FIG_DIR / 'fa_band_comparison.pdf')
display(fig)
plt.close(fig)

fa_rows = []
for name, data in CASE_DATA.items():
    for q2_value in (0.05, 0.2, 0.5, 1.0):
        k = int(np.argmin(np.abs(Q2 - q2_value)))
        fa_rows.append({'case': data['title'], 'Q2 [GeV^2]': Q2[k],
                        '-F_A median (before)': -data['fa_before'][1][k], '-F_A median (after)': -data['fa_after'][1][k],
                        '68% half-width (before)': (data['fa_before'][2][k] - data['fa_before'][0][k]) / 2,
                        '68% half-width (after)': (data['fa_after'][2][k] - data['fa_after'][0][k]) / 2,
                        'median shift [%]': 100 * (data['fa_after'][1][k] / data['fa_before'][1][k] - 1),
                        'half-width change [%]': 100 * ((data['fa_after'][2][k] - data['fa_after'][0][k]) / (data['fa_before'][2][k] - data['fa_before'][0][k]) - 1)})
fa_table = pd.DataFrame(fa_rows).set_index(['case', 'Q2 [GeV^2]'])
display(fa_table.style.format({'-F_A median (before)': '{:.5f}', '-F_A median (after)': '{:.5f}', '68% half-width (before)': '{:.5f}',
                               '68% half-width (after)': '{:.5f}', 'median shift [%]': '{:+.3f}', 'half-width change [%]': '{:+.2f}'}))
if SAVE_OUTPUTS:
    fa_table.to_csv(TABLE_DIR / 'fa_band_comparison.csv')

## Section 6: importance-sampling diagnostics for every z-expansion fit

Two diagnostics per stored chain, from the weights $w_k=\exp(+\Delta\chi^2_{\rm data}(\eta^{(k)})/2)$:

- **ESS / N**, the Kish effective sample size $\left(\sum_k w_k\right)^2/\sum_k w_k^2$ divided by the chain length. It says how much of the chain survives the reweighting.
- **$\hat{k}$**, the shape parameter of a generalized Pareto distribution fitted to the tail of the weights (PSIS, Vehtari et al., arXiv:1507.02646; the tail is the largest $\min(N/5,\,3\sqrt{N})$ weights, as in `arviz.psislw`). $\hat{k}<0.5$ means the weights have finite variance, $0.5\le\hat{k}<0.7$ is still usable, $\hat{k}\ge0.7$ means the reweighted estimates should not be trusted. ESS alone can look healthy while a single sample dominates the tail; $\hat{k}$ is what catches that.

`pareto_khat` in `spline_reweighting.py` is a transcription of the reference implementation and reproduces `arviz.psislw` exactly on these chains (`arviz` itself is not a dependency of this repo: the version that installs on this Python cannot be imported against the venv's SciPy). Each fit has a chain in all three data suites, so the table gives both diagnostics for each.

In [ ]:
DIAGNOSTIC_FITS = ['minerva_k8', 'minerva_k7', 'minerva_k6', 'lqcd_k6', 'minerva_lqcd_k6',
                   'minerva_k6_nuisance', 'minerva_k6_uniform', 'minerva_k6_uniform_nuisance']
DIAGNOSTIC_SUITES = ('nuwro', 'asimov', 'opendata')

diagnostic_rows, no_chain = [], []
for fit in DIAGNOSTIC_FITS:
    spec = SPEC_BY_KEY[fit]
    row = {'fit': fit, 'k_max': spec.prior.kmax}
    for suite in DIAGNOSTIC_SUITES:
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            _, eta = load_chain(fit, suite)
            if eta is None:
                no_chain.append(f'{fit}/{suite}')
                row[(SUITE_LABEL[suite], 'ESS / N')] = row[(SUITE_LABEL[suite], 'k-hat')] = np.nan
                continue
            grid = load_dchi2_grid(fit, suite)
            _, _, ess, dchi2 = compute_importance_weights(eta, grid, verbose=False)
        # log w_k = +dchi2_data(eta_k)/2 by construction; both diagnostics are invariant
        # under the overall normalization of the weights.
        row[(SUITE_LABEL[suite], 'ESS / N')] = ess / len(eta)
        row[(SUITE_LABEL[suite], 'k-hat')] = pareto_khat(0.5 * dchi2)
    diagnostic_rows.append(row)

importance_diagnostics = pd.DataFrame(diagnostic_rows).set_index(['fit', 'k_max'])
importance_diagnostics.columns = pd.MultiIndex.from_tuples(importance_diagnostics.columns)
display(importance_diagnostics.style.format('{:.3f}', na_rep='-'))

ess_columns = [c for c in importance_diagnostics.columns if c[1] == 'ESS / N']
khat_columns = [c for c in importance_diagnostics.columns if c[1] == 'k-hat']
lowest_ess = importance_diagnostics[ess_columns].min().min()
largest_khat = importance_diagnostics[khat_columns].max().max()
print(f'Lowest ESS / N over all fits and suites: {lowest_ess:.3f}; largest k-hat: {largest_khat:.3f} '
      f'(k-hat < 0.5: finite weight variance; 0.5-0.7: usable; >= 0.7: unreliable).')
if no_chain:
    print('No stored chain, nothing to diagnose:', ', '.join(no_chain))
assert largest_khat < 0.7, f'PSIS k-hat = {largest_khat:.3f} >= 0.7: the reweighting is not reliable'
if SAVE_OUTPUTS:
    importance_diagnostics.to_csv(TABLE_DIR / 'importance_diagnostics.csv')
    print('Saved:', TABLE_DIR / 'importance_diagnostics.csv')